# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vinod5566773/flyrank-capstone/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Two paper findings + my methodology questions

**Finding 1 — Content performance can decline over time**

My analysis labels an item as declining when its latest-window impressions are below 80% of its previous-window impressions. This is an operational definition for my project, not proof of why the decline happened. A validation question is whether the two measurement windows are comparable and whether missing data, seasonality, or tracking changes could explain some of the observed decline.

**Finding 2 — Multiple signals can support review prioritization**

My analysis combines previous impressions and query-derived signals to identify content items for review. The client-held-out Random Forest achieved 0.611 accuracy and 0.704 class-1 F1, compared with 0.677 accuracy for the majority-class baseline. This suggests the current model has limitations in generalizing to unseen clients. A stronger claim would require validation on suitable held-out data and comparison against a baseline using the same split and metrics.

**Methodology questions**

* Are the measurement windows strictly defined and comparable?
* Are all model features available at the time a review decision would be made?
* Does the evaluation split prevent information from the same client appearing in both training and testing?
* Do the results generalize to future periods and other clients?


In [1]:
print("Finding 1: Decline is defined using two impression windows.")
print("Finding 2: Multiple signals are used for review prioritization.")
print("Validation questions: window comparability, feature timing,")
print("client separation, and generalization to future periods.")

Finding 1: Decline is defined using two impression windows.
Finding 2: Multiple signals are used for review prioritization.
Validation questions: window comparability, feature timing,
client separation, and generalization to future periods.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### My model under an honest split (before/after)

I compared a random row split with a client-grouped split. The random split achieved 0.656 accuracy and 0.756 class-1 F1. Under the client-held-out split, accuracy was 0.611 and class-1 F1 was 0.704. The majority-class baseline for the grouped split achieved 0.677 accuracy.

This comparison shows that the evaluation design affects the measured results. The client-held-out model did not beat the majority-class baseline on accuracy. A separate experiment adding `kept_impressions` reached 0.663 accuracy and 0.745 class-1 F1, but its accuracy was still below the grouped-split baseline.

These are previously measured results, not a new training run in this notebook. A reproducible validation should train and evaluate the model using the same data, feature set, split, and metrics.


In [2]:
import pandas as pd

validation_comparison = pd.DataFrame([
    {
        "experiment": "Random row split",
        "accuracy": 0.656,
        "class_1_f1": 0.756,
    },
    {
        "experiment": "Client-held-out Random Forest",
        "accuracy": 0.611,
        "class_1_f1": 0.704,
    },
    {
        "experiment": "Client-held-out + kept_impressions",
        "accuracy": 0.663,
        "class_1_f1": 0.745,
    },
])

display(validation_comparison)

print("Client-held-out majority-class baseline accuracy: 0.677")

,experiment,accuracy,class_1_f1
0,Random row split,0.656,0.756
1,Client-held-out Random Forest,0.611,0.704
2,Client-held-out + kept_impressions,0.663,0.745


Client-held-out majority-class baseline accuracy: 0.677


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

My target is `is_declining`, defined as latest-window impressions being below 80% of previous-window impressions.

* **Identifiers:** `client_hash_id` and `content_hash_id` are context/grouping fields, not model features.
* **Target overlap:** `imp_last30` directly defines the target, so it should not be used as a feature when predicting the target.
* **Previous-window features:** `imp_prev30`, `clk_last30`, and `pos_last30` may be usable if they were available before the review decision.
* **Query-derived features:** `visible_queries`, `rare_share`, `anon_share`, and `top_query_share` require confirmation that they were calculated only from information available at prediction time.
* **Kept impressions:** `kept_impressions` improved measured metrics in a separate experiment, but its timing must be verified before including it in a predictive model.

The audit identifies possible leakage risks; it does not establish that every feature is leakage-free. Feature definitions and timestamps should be checked before making that claim.


In [3]:
identifier_fields = ["client_hash_id", "content_hash_id"]
target_field = "is_declining"
target_derived_fields = ["imp_last30"]

candidate_features = [
    "imp_prev30",
    "clk_last30",
    "pos_last30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share",
    "kept_impressions",
]

assert target_field not in candidate_features
assert "imp_last30" not in candidate_features
assert not any(field in candidate_features for field in identifier_fields)

print("Leakage checks passed for identifiers and direct target overlap.")
print("Feature timing still needs verification before predictive use.")
print("Candidate features:", candidate_features)

Leakage checks passed for identifiers and direct target overlap.
Feature timing still needs verification before predictive use.
Candidate features: ['imp_prev30', 'clk_last30', 'pos_last30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share', 'kept_impressions']


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Original claim:** The AI model identifies content that needs refreshing and helps improve content performance.

**Safer rewritten claim:** In this dataset, the model combines historical impression and query-derived signals to rank content items for human review. The client-held-out experiment measured 0.611 accuracy and 0.704 class-1 F1, while the majority-class baseline achieved 0.677 accuracy. These results indicate that the current model has limitations in generalizing to unseen clients. The ranked list is a decision-support tool for investigation; it does not prove that refreshing a selected item will improve its performance.


In [4]:
original_claim = (
    "The AI model identifies content that needs refreshing "
    "and improves content performance."
)

safer_claim = (
    "The model ranks content items for human review using historical signals. "
    "Its client-held-out accuracy was 0.611 versus 0.677 for the baseline. "
    "The results support investigation, not causal claims about refresh impact."
)

print("Original claim:", original_claim)
print("\nSafer claim:", safer_claim)

Original claim: The AI model identifies content that needs refreshing and improves content performance.

Safer claim: The model ranks content items for human review using historical signals. Its client-held-out accuracy was 0.611 versus 0.677 for the baseline. The results support investigation, not causal claims about refresh impact.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.